In [ ]:
from __future__ import annotations

import argparse
import math
import multiprocessing as mp
import os
import time
from pathlib import Path
from typing import List, Optional, Tuple

import cv2
import numpy as np
import trimesh


DEFAULT_INPUT_ROOT = r"./Held_Out3.stl"
DEFAULT_OUTPUT_ROOT = r"./out"

DEFAULTS = dict(
    azim_steps=1,
    polar_steps=1,
    azim=(150.0, 150.0),
    polar=(63.0, 63.0),
    radius=54.435,
    res=(640, 640),
    focal_mm=32,
    sensor_width=36.0,
    gray_prefix="gray_",
    depth_prefix="depth_",
    normal_prefix="normal_",
    depth_gamma=1.3,
    depth_min_span_scale=0.9,
)


FLAT_GRAY_LINEAR = 0.72


LINE_COLOR_LINEAR = 0.0
LINE_THICKNESS_PX = 1.25
LINE_SOFTNESS_PX = 0.25


FREESTYLE_CREASE_ANGLE_DEG = 134.43006203652013


SUPERSAMPLE = 4


EDGE_DEPTH_TOL_FRAC = 0.01

EDGE_VIS_SAMPLES = 24


BONE_RAMP_STOPS = [
    (0.00, (0.00, 0.00, 0.00)),
    (0.12, (0.05, 0.06, 0.12)),
    (0.38, (0.22, 0.27, 0.40)),
    (0.68, (0.60, 0.70, 0.78)),
    (0.86, (0.82, 0.90, 0.94)),
    (1.00, (1.00, 1.00, 1.00)),
]

In [ ]:
def linear_to_srgb(x: np.ndarray) -> np.ndarray:

    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.0031308, x * 12.92, 1.055 * np.power(x, 1.0 / 2.4) - 0.055)


class MeshData:


    def __init__(self, path: Path) -> None:
        tm = trimesh.load(path, force="mesh")
        if isinstance(tm, trimesh.Scene):
            tm = trimesh.util.concatenate(
                [g for g in tm.geometry.values() if len(g.faces) > 0]
            )
        if tm.is_empty or len(tm.faces) == 0:
            raise ValueError(f"Failed to load / empty mesh: {path}")


        tm.fix_normals()

        V = np.asarray(tm.vertices, dtype=np.float64)
        F = np.asarray(tm.faces, dtype=np.int64)


        lo, hi = V.min(axis=0), V.max(axis=0)
        V = V - np.array([(lo[0] + hi[0]) * 0.5, (lo[1] + hi[1]) * 0.5, lo[2]])


        lo, hi = V.min(axis=0), V.max(axis=0)
        self.center = (lo + hi) * 0.5
        self.diag = float(np.linalg.norm(hi - lo))
        self.sphere_r = 0.5 * self.diag

        self.vertices = V
        self.faces = F
        self.face_normals = np.asarray(tm.face_normals, dtype=np.float64)
        self.face_centroids = V[F].mean(axis=1)


        adj = np.asarray(tm.face_adjacency, dtype=np.int64)
        adj_edges = np.asarray(tm.face_adjacency_edges, dtype=np.int64)
        adj_angles = np.asarray(tm.face_adjacency_angles, dtype=np.float64)

        self.adj = adj
        self.adj_edges = adj_edges

        self.crease_mask = adj_angles > math.radians(180.0 - FREESTYLE_CREASE_ANGLE_DEG)


        e = np.sort(np.asarray(tm.edges, dtype=np.int64), axis=1)
        uniq, counts = np.unique(e, axis=0, return_counts=True)
        self.border_edges = uniq[counts == 1]

    def auto_radius(self) -> float:
        return max(2.2 * self.sphere_r, self.sphere_r + 0.5)


def spherical_to_cartesian(r: float, elev_deg: float, azim_deg: float) -> np.ndarray:
    e = math.radians(elev_deg)
    a = math.radians(azim_deg)
    return np.array(
        [r * math.cos(e) * math.cos(a), r * math.cos(e) * math.sin(a), r * math.sin(e)],
        dtype=np.float64,
    )


def camera_basis(eye: np.ndarray, target: np.ndarray, polar: float, azim: float):


    if abs(float(polar)) < 1e-6:
        ang = math.radians(float(azim) + 90.0)
        c, s = math.cos(ang), math.sin(ang)
        right = np.array([c, s, 0.0])
        up = np.array([-s, c, 0.0])
        back = np.array([0.0, 0.0, 1.0])
    else:
        f = target - eye
        f /= np.linalg.norm(f)
        world_up = np.array([0.0, 0.0, 1.0])
        right = np.cross(f, world_up)
        right /= np.linalg.norm(right)
        up = np.cross(right, f)
        back = -f
    return np.stack([right, up, back], axis=0)


def project(V: np.ndarray, R_wc: np.ndarray, eye: np.ndarray,
            focal_px: float, width: int, height: int):

    cam = (V - eye) @ R_wc.T
    depth = -cam[:, 2]
    safe = np.maximum(depth, 1e-9)
    x = focal_px * (cam[:, 0] / safe) + width * 0.5 - 0.5
    y = -focal_px * (cam[:, 1] / safe) + height * 0.5 - 0.5
    return np.stack([x, y], axis=1), depth


def rasterize(screen: np.ndarray, depth: np.ndarray, faces: np.ndarray,
              width: int, height: int):

    zbuf = np.full((height, width), np.inf, dtype=np.float64)
    fid = np.full((height, width), -1, dtype=np.int32)

    tri_s = screen[faces]
    tri_z = depth[faces]

    valid = np.all(tri_z > 1e-9, axis=1)
    mins = np.floor(tri_s.min(axis=1)).astype(np.int64)
    maxs = np.ceil(tri_s.max(axis=1)).astype(np.int64)
    on_screen = (maxs[:, 0] >= 0) & (maxs[:, 1] >= 0) & \
                (mins[:, 0] <= width - 1) & (mins[:, 1] <= height - 1)
    todo = np.nonzero(valid & on_screen)[0]

    for f in todo:
        p = tri_s[f]
        z = tri_z[f]
        x0 = max(int(mins[f, 0]), 0)
        y0 = max(int(mins[f, 1]), 0)
        x1 = min(int(maxs[f, 0]), width - 1)
        y1 = min(int(maxs[f, 1]), height - 1)
        if x1 < x0 or y1 < y0:
            continue

        xs = np.arange(x0, x1 + 1)
        ys = np.arange(y0, y1 + 1)
        gx, gy = np.meshgrid(xs, ys)

        ax, ay = p[0]
        bx, by = p[1]
        cx, cy = p[2]
        den = (by - cy) * (ax - cx) + (cx - bx) * (ay - cy)
        if abs(den) < 1e-12:
            continue
        inv = 1.0 / den
        w0 = ((by - cy) * (gx - cx) + (cx - bx) * (gy - cy)) * inv
        w1 = ((cy - ay) * (gx - cx) + (ax - cx) * (gy - cy)) * inv
        w2 = 1.0 - w0 - w1

        inside = (w0 >= 0) & (w1 >= 0) & (w2 >= 0)
        if not inside.any():
            continue


        iz = w0 / z[0] + w1 / z[1] + w2 / z[2]
        with np.errstate(divide="ignore", invalid="ignore"):
            zz = np.where(iz > 0, 1.0 / iz, np.inf)

        sub = inside & (zz < zbuf[y0:y1 + 1, x0:x1 + 1])
        if not sub.any():
            continue
        zbuf[y0:y1 + 1, x0:x1 + 1][sub] = zz[sub]
        fid[y0:y1 + 1, x0:x1 + 1][sub] = f

    return zbuf, fid


def visible_edge_segments(md: MeshData, screen: np.ndarray, depth: np.ndarray,
                          zbuf: np.ndarray, edge_pairs: np.ndarray,
                          width: int, height: int, tol: float):


    if len(edge_pairs) == 0:
        return []

    v0 = edge_pairs[:, 0]
    v1 = edge_pairs[:, 1]
    p0 = screen[v0]
    p1 = screen[v1]
    z0 = depth[v0]
    z1 = depth[v1]

    keep = (z0 > 1e-9) & (z1 > 1e-9)
    if not keep.any():
        return []

    t = np.linspace(0.0, 1.0, EDGE_VIS_SAMPLES)[None, :]
    px = p0[:, 0:1] * (1 - t) + p1[:, 0:1] * t
    py = p0[:, 1:2] * (1 - t) + p1[:, 1:2] * t

    inv = (1.0 / z0)[:, None] * (1 - t) + (1.0 / z1)[:, None] * t
    pz = 1.0 / np.maximum(inv, 1e-12)

    ix = np.clip(np.round(px).astype(np.int64), 0, width - 1)
    iy = np.clip(np.round(py).astype(np.int64), 0, height - 1)
    zb = zbuf[iy, ix]

    on_screen = (px >= -1) & (px <= width) & (py >= -1) & (py <= height)
    vis = on_screen & (~np.isfinite(zb) | (pz <= zb + tol))
    vis &= keep[:, None]

    segments = []
    for i in np.nonzero(vis.any(axis=1))[0]:
        row = vis[i]

        idx = np.nonzero(row)[0]
        splits = np.split(idx, np.nonzero(np.diff(idx) != 1)[0] + 1)
        for run in splits:
            if run.size == 0:
                continue
            a, b = run[0], run[-1]
            if a == b:
                if run.size == 1 and EDGE_VIS_SAMPLES > 1:

                    b = min(a + 1, EDGE_VIS_SAMPLES - 1)
            segments.append(((px[i, a], py[i, a]), (px[i, b], py[i, b])))
    return segments


def stroke_coverage(segs, width: int, height: int,
                    half_width: float, softness: float) -> np.ndarray:


    cov = np.zeros((height, width), dtype=np.float64)
    reach = half_width + 3.0 * softness + 1.0
    for (ax, ay), (bx, by) in segs:
        x0 = max(0, int(np.floor(min(ax, bx) - reach)))
        x1 = min(width - 1, int(np.ceil(max(ax, bx) + reach)))
        y0 = max(0, int(np.floor(min(ay, by) - reach)))
        y1 = min(height - 1, int(np.ceil(max(ay, by) + reach)))
        if x1 < x0 or y1 < y0:
            continue
        gx, gy = np.meshgrid(np.arange(x0, x1 + 1, dtype=np.float64),
                             np.arange(y0, y1 + 1, dtype=np.float64))
        vx, vy = bx - ax, by - ay
        L2 = vx * vx + vy * vy
        if L2 < 1e-12:
            t = np.zeros_like(gx)
        else:
            t = np.clip(((gx - ax) * vx + (gy - ay) * vy) / L2, 0.0, 1.0)
        d = np.hypot(gx - (ax + t * vx), gy - (ay + t * vy))

        c = np.clip(half_width + 0.5 - d, 0.0, 1.0)
        block = cov[y0:y1 + 1, x0:x1 + 1]
        np.maximum(block, c, out=block)
    if softness > 0:
        k = int(softness * 6) | 1
        cov = cv2.GaussianBlur(cov, (k, k), sigmaX=softness)
    return np.clip(cov, 0.0, 1.0)


def apply_bone_ramp(t: np.ndarray) -> np.ndarray:

    t = np.clip(t, 0.0, 1.0)
    out = np.zeros(t.shape + (3,), dtype=np.float64)
    stops = BONE_RAMP_STOPS
    for (p0, c0), (p1, c1) in zip(stops[:-1], stops[1:]):
        seg = (t >= p0) & (t <= p1)
        span = max(p1 - p0, 1e-12)
        f = np.clip((t[seg] - p0) / span, 0.0, 1.0)
        c0a, c1a = np.array(c0), np.array(c1)
        out[seg] = c0a[None, :] * (1 - f)[:, None] + c1a[None, :] * f[:, None]
    return out


def depth_range_for_view(md: MeshData, R_wc: np.ndarray, eye: np.ndarray,
                         vertex_depth: np.ndarray, min_span_scale: float):


    eps = 1e-4
    v = vertex_depth[vertex_depth > eps]
    lo, hi = md.vertices.min(axis=0), md.vertices.max(axis=0)
    corners = np.array([[x, y, z] for x in (lo[0], hi[0])
                        for y in (lo[1], hi[1]) for z in (lo[2], hi[2])])
    cam_c = (corners - eye) @ R_wc.T
    b = -cam_c[:, 2]
    b = b[b > eps]

    candidates = np.concatenate([v, b]) if v.size and b.size else (v if v.size else b)
    if candidates.size == 0:
        z_from, z_to = eps, 1.0 + eps
    else:
        z_from, z_to = float(candidates.min()), float(candidates.max())

    span = max(1e-6, z_to - z_from)
    min_span = max(1e-3, min_span_scale * md.sphere_r)
    if span < min_span:
        mid = 0.5 * (z_from + z_to)
        z_from, z_to = mid - 0.5 * min_span, mid + 0.5 * min_span

    span = z_to - z_from
    z_from -= 0.01 * span
    z_to += 0.01 * span
    z_from = max(eps, z_from)
    z_to = max(z_from + 1e-4, z_to)
    return z_from, z_to


def render_view(md: MeshData, polar: float, azim: float, radius: float,
                focal_mm: float, sensor_width: float,
                out_w: int, out_h: int, supersample: int,
                depth_gamma: float = DEFAULTS["depth_gamma"],
                depth_min_span_scale: float = DEFAULTS["depth_min_span_scale"]) -> dict:
    W, H = out_w * supersample, out_h * supersample

    elev = 90.0 - float(polar)
    eye = md.center + spherical_to_cartesian(radius, elev, azim)
    R_wc = camera_basis(eye, md.center, polar, azim)


    focal_px = (focal_mm / sensor_width) * max(W, H)

    screen, depth = project(md.vertices, R_wc, eye, focal_px, W, H)
    zbuf, fid = rasterize(screen, depth, md.faces, W, H)
    mask = fid >= 0


    gray = np.ones((H, W), dtype=np.float64)
    gray[mask] = FLAT_GRAY_LINEAR


    view_vec = eye[None, :] - md.face_centroids
    front = np.einsum("ij,ij->i", md.face_normals, view_vec) > 0.0
    sil = front[md.adj[:, 0]] != front[md.adj[:, 1]]
    pairs = md.adj_edges[sil | md.crease_mask]
    if len(md.border_edges):
        pairs = np.vstack([pairs, md.border_edges])

    tol = EDGE_DEPTH_TOL_FRAC * md.sphere_r
    segs = visible_edge_segments(md, screen, depth, zbuf, pairs, W, H, tol)

    if segs:
        coverage = stroke_coverage(segs, W, H,
                                   LINE_THICKNESS_PX * supersample * 0.5,
                                   LINE_SOFTNESS_PX * supersample)
        gray = gray * (1.0 - coverage) + LINE_COLOR_LINEAR * coverage


    z_from, z_to = depth_range_for_view(md, R_wc, eye, depth, depth_min_span_scale)
    t = np.zeros((H, W), dtype=np.float64)
    t[mask] = np.clip((zbuf[mask] - z_from) / (z_to - z_from), 0.0, 1.0)
    t = np.power(t, max(1e-3, depth_gamma))
    depth_rgb = apply_bone_ramp(t)
    white = np.ones((H, W, 3), dtype=np.float64)
    depth_img = np.where(mask[:, :, None], depth_rgb, white)


    R_normal = R_wc.copy()
    R_normal[2] *= -1.0
    normal_img = np.ones((H, W, 3), dtype=np.float64)
    if mask.any():
        face_n_cam = md.face_normals[fid[mask]] @ R_normal.T
        normal_img[mask] = np.clip(face_n_cam * 0.5 + 0.5, 0.0, 1.0)


    out = {}
    for key, im in (("grayscale", gray), ("depth", depth_img), ("normal", normal_img)):
        if supersample > 1:
            im = cv2.resize(im, (out_w, out_h), interpolation=cv2.INTER_AREA)
        out[key] = linear_to_srgb(im)
    return out


def save_16(path: Path, img: np.ndarray) -> None:


    path.parent.mkdir(parents=True, exist_ok=True)
    u16 = np.clip(np.rint(img * 65535.0), 0, 65535).astype(np.uint16)
    if u16.ndim == 2:
        u16 = np.repeat(u16[:, :, None], 3, axis=2)
    else:
        u16 = u16[:, :, ::-1]
    cv2.imwrite(str(path), u16)


def angle_tag(azim: float, polar: float) -> str:
    return f"az{int(round(azim)) % 360:03d}_pol{int(round(polar)) % 180:03d}"


_W = {}
PASS_PREFIX_KEYS = {"grayscale": "gray_prefix", "depth": "depth_prefix", "normal": "normal_prefix"}


def _init_worker(md, radius, focal_mm, sensor_width, res, supersample, out_dir,
                 prefixes, depth_gamma, depth_min_span_scale):
    _W.update(md=md, radius=radius, focal_mm=focal_mm, sensor_width=sensor_width,
              res=res, supersample=supersample, out_dir=out_dir, prefixes=prefixes,
              depth_gamma=depth_gamma, depth_min_span_scale=depth_min_span_scale)


def _render_and_save(view):
    polar, azim = view
    imgs = render_view(_W["md"], polar, azim, _W["radius"], _W["focal_mm"],
                       _W["sensor_width"], _W["res"][0], _W["res"][1],
                       _W["supersample"], _W["depth_gamma"], _W["depth_min_span_scale"])
    tag = angle_tag(azim, polar)
    for pass_name, img in imgs.items():
        name = f"{_W['prefixes'][pass_name]}{tag}.png"
        save_16(Path(_W["out_dir"]) / pass_name / name, img)
    return view


def build_views(azim_range, polar_range, azim_steps, polar_steps):

    if azim_steps <= 0:
        azim_steps = 12
    if polar_steps <= 0:
        polar_steps = 11
    az0, az1 = azim_range
    p0, p1 = polar_range
    if abs((az1 - az0) - 360.0) < 1e-6:
        azs = np.linspace(az0, az1, azim_steps + 1, endpoint=True)[:-1]
    else:
        azs = np.linspace(az0, az1, azim_steps, endpoint=True)
    polars = np.linspace(p0, p1, polar_steps, endpoint=True)
    return [(float(pol), float(az)) for az in azs for pol in polars]


def generate_images_for_one_stl(obj_path: str, object_output_dir: str, radius: float,
                                polar_range, azim_range, polar_steps: int,
                                azim_steps: int, res, focal_mm: float,
                                sensor_width: float, gray_prefix: str,
                                depth_prefix: str, normal_prefix: str,
                                depth_gamma: float, depth_min_span_scale: float,
                                supersample: int, workers: int) -> None:
    md = MeshData(Path(obj_path))
    R = radius if (radius and radius > 0.0) else md.auto_radius()

    views = build_views(azim_range, polar_range, azim_steps, polar_steps)
    prefixes = {"grayscale": gray_prefix, "depth": depth_prefix, "normal": normal_prefix}
    for pass_name in prefixes:
        Path(object_output_dir, pass_name).mkdir(parents=True, exist_ok=True)

    print(f"       {len(md.vertices):,} verts, {len(md.faces):,} faces, "
          f"sphere_r {md.sphere_r:.4f}, R {R:.4f}, {len(views)} views")

    t0 = time.time()
    args = (md, R, focal_mm, sensor_width, res, supersample, object_output_dir,
            prefixes, depth_gamma, depth_min_span_scale)
    if workers > 1:
        ctx = mp.get_context("fork")
        with ctx.Pool(workers, initializer=_init_worker, initargs=args) as pool:
            for _ in pool.imap_unordered(_render_and_save, views, chunksize=1):
                pass
    else:
        _init_worker(*args)
        for v in views:
            _render_and_save(v)
    print(f"[INFO] Wrote {len(views)} x 3-pass images to: {object_output_dir} "
          f"({time.time() - t0:.1f}s)")


def list_stls_recursive(input_root: str) -> List[str]:
    root = Path(input_root)
    if root.is_file() and root.suffix.lower() == ".stl":
        return [str(root)]
    return [str(p) for p in root.rglob("*.stl")]


def run_batch(input_root, output_root, radius, polar_range, azim_range, polar_steps,
              azim_steps, res, focal_mm, sensor_width, gray_prefix, depth_prefix,
              normal_prefix, depth_gamma, depth_min_span_scale, supersample,
              workers) -> None:
    stls = list_stls_recursive(input_root)
    if not stls:
        raise RuntimeError(f"No STL files found under: {input_root}")
    Path(output_root).mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Found {len(stls)} STL(s) under: {input_root}")
    print(f"[INFO] Output root: {output_root}")
    for i, stl in enumerate(stls, 1):
        object_output_dir = str(Path(output_root) / Path(stl).stem)
        print(f"[INFO] ({i}/{len(stls)}) {stl}")
        print(f"       -> {object_output_dir}")
        generate_images_for_one_stl(
            stl, object_output_dir, radius, polar_range, azim_range, polar_steps,
            azim_steps, res, focal_mm, sensor_width, gray_prefix, depth_prefix,
            normal_prefix, depth_gamma, depth_min_span_scale, supersample, workers,
        )


def parse_args(argv: Optional[List[str]] = None):
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--input_root", default=DEFAULT_INPUT_ROOT)
    p.add_argument("--output_root", default=DEFAULT_OUTPUT_ROOT)
    p.add_argument("--azim_steps", type=int, default=DEFAULTS["azim_steps"])
    p.add_argument("--polar_steps", type=int, default=DEFAULTS["polar_steps"])
    p.add_argument("--azim", nargs=2, type=float, default=list(DEFAULTS["azim"]))
    p.add_argument("--polar", nargs=2, type=float, default=list(DEFAULTS["polar"]))
    p.add_argument("--radius", type=float, default=DEFAULTS["radius"])
    p.add_argument("--res", nargs=2, type=int, default=list(DEFAULTS["res"]))
    p.add_argument("--focal_mm", type=float, default=DEFAULTS["focal_mm"])
    p.add_argument("--sensor_width", type=float, default=DEFAULTS["sensor_width"])
    p.add_argument("--gray_prefix", default=DEFAULTS["gray_prefix"])
    p.add_argument("--depth_prefix", default=DEFAULTS["depth_prefix"])
    p.add_argument("--normal_prefix", default=DEFAULTS["normal_prefix"])
    p.add_argument("--depth_gamma", type=float, default=DEFAULTS["depth_gamma"])
    p.add_argument("--depth_min_span_scale", type=float,
                   default=DEFAULTS["depth_min_span_scale"])
    p.add_argument("--supersample", type=int, default=SUPERSAMPLE,
                   help="Anti-aliasing factor (stand-in for Cycles' AA samples).")
    p.add_argument("--workers", type=int, default=min(16, os.cpu_count() or 1))
    return p.parse_args(argv)

In [ ]:
args = parse_args([])
args.workers = 1

run_batch(
    input_root=args.input_root,
    output_root=args.output_root,
    radius=args.radius,
    polar_range=tuple(args.polar),
    azim_range=tuple(args.azim),
    polar_steps=args.polar_steps,
    azim_steps=args.azim_steps,
    res=tuple(args.res),
    focal_mm=args.focal_mm,
    sensor_width=args.sensor_width,
    gray_prefix=args.gray_prefix,
    depth_prefix=args.depth_prefix,
    normal_prefix=args.normal_prefix,
    depth_gamma=args.depth_gamma,
    depth_min_span_scale=args.depth_min_span_scale,
    supersample=args.supersample,
    workers=args.workers,
)
